In [44]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, get_linear_schedule_with_warmup
import json
import random
import numpy as np
import math

In [45]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [46]:
tokenizer = AutoTokenizer.from_pretrained("roberta-base")
model = AutoModelForSequenceClassification.from_pretrained("roberta-base", num_labels=5).to(device)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [47]:
def generate_dataset():
    np.random.seed(42)
    random.seed(42)

    categories = ["TECH", "POLITICS", "SPORT", "ENTERTAINMENT", "SCIENCE"]
    # Asymetria klas
    class_weights = [0.45, 0.25, 0.15, 0.10, 0.05]

    vocab_per_cat = {
        "TECH": ["gpu", "cuda", "kernel", "latency", "transformer", "dram", "quantization", "tensor"],
        "POLITICS": ["election", "parliament", "bill", "vote", "sanction", "diplomacy", "treaty", "minister"],
        "SPORT": ["stadium", "referee", "offside", "tournament", "championship", "match", "penalty", "score"],
        "ENTERTAINMENT": ["cinema", "actor", "boxoffice", "premiere", "soundtrack", "director", "sequel", "album"],
        "SCIENCE": ["quantum", "supernova", "enzyme", "hypothesis", "spectrometry", "genome", "photon", "particle"]
    }

    noise_chars = ["#", "$", "@", "*", "!", "%"]

    def make_sample():
        cat = np.random.choice(categories, p=class_weights)
        words = np.random.choice(vocab_per_cat[cat], size=random.randint(15, 45)).tolist()

        # Wstrzykiwanie szumu tekstowego (simulated OCR / scraping noise)
        for i in range(len(words)):
            if random.random() < 0.2:
                idx = random.randint(0, len(words[i]))
                words[i] = words[i][:idx] + random.choice(noise_chars) + words[i][idx:]

        text = " ".join(words)
        return text, categories.index(cat)

    train_data = [{"text": text, "label": label} for text, label in [make_sample() for _ in range(2500)]]
    test_data = [{"text": text, "label": label} for text, label in [make_sample() for _ in range(500)]]

    return train_data, test_data

    print("Wygenerowano pliki train.json (2500 próbek) i test.json (500 próbek).")

if __name__ == "__main__":
    train_data, test_data = generate_dataset()

In [48]:
X = [q["text"] for q in train_data]

X_tokenized = tokenizer(X, truncation=True, padding=True, return_tensors="pt")

y = torch.tensor([q["label"] for q in train_data], dtype=torch.long)


In [49]:
class MyDataset(torch.utils.data.Dataset):
  def __init__(self, input_ids, attention_mask, labels):
    super().__init__()
    self.input_ids = input_ids
    self.att_mask = attention_mask
    self.labels = labels

  def __len__(self):
    return len(self.labels)

  def __getitem__(self, idx):
    return {"input_ids": self.input_ids[idx].to(device), "attention_mask": self.att_mask[idx].to(device), "labels": self.labels[idx].to(device)}

In [50]:
epochs = 4
LR = 2e-5
acum_steps = 16

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, fused=True)
criterion = torch.nn.CrossEntropyLoss(weight=torch.tensor([0.45, 0.25, 0.15, 0.10, 0.05], dtype=torch.float32)).to(device)
scaler = torch.amp.GradScaler('cuda')

collator = DataCollatorWithPadding(tokenizer=tokenizer)

dataset = MyDataset(X_tokenized["input_ids"], X_tokenized["attention_mask"], y)
dataloader = torch.utils.data.DataLoader(dataset, shuffle=True, batch_size=16, collate_fn=collator)

steps_per_epoch = math.ceil(len(dataloader) / acum_steps)
t_total = steps_per_epoch * epochs

scheduler = get_linear_schedule_with_warmup(optimizer, int(t_total*0.1), t_total)

for epoch in range(epochs):
  total_loss = 0
  optimizer.zero_grad()

  for i, batch in enumerate(dataloader):
    batch = batch.to(device)
    with torch.amp.autocast("cuda", enabled=(device=="cuda")):
      out = model(**batch)

      #print(out["logits"])
      preds = out["logits"].to(device)

      loss = criterion(preds, batch["labels"])
      loss = loss / acum_steps

    scaler.scale(loss).backward()


    if (i+1) % acum_steps == 0 or i == len(dataloader)-1:
      scaler.unscale_(optimizer)

      torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
      scaler.step(optimizer)

      scaler.update()
      scheduler.step()

      optimizer.zero_grad()

    total_loss += loss.item() * acum_steps


  print(f"Epoch: {epoch}, loss: {total_loss/len(dataloader)}")

  if torch.cuda.is_available():
    torch.cuda.empty_cache()


Epoch: 0, loss: 1.2747554524688964
Epoch: 1, loss: 0.5758770306588737
Epoch: 2, loss: 0.23132270617280037
Epoch: 3, loss: 0.10319069336364224


In [51]:
from sklearn.metrics import f1_score

def evaluate_model(model, tokenizer, test_data, device):
    model.eval()

    X_test = [q["text"] for q in test_data]
    y_test = torch.tensor([q["label"] for q in test_data], dtype=torch.long)

    X_test_tokenized = tokenizer(X_test, truncation=True, padding=True, return_tensors="pt")

    test_dataset = MyDataset(X_test_tokenized["input_ids"], X_test_tokenized["attention_mask"], y_test)
    test_dataloader = torch.utils.data.DataLoader(test_dataset, shuffle=False, batch_size=16, collate_fn=collator) # Use the same collator as training

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in test_dataloader:
            batch = batch.to(device)
            # MyDataset already moves tensors to device in __getitem__
            # batch = {k: v.to(device) for k, v in batch.items()}

            outputs = model(**batch)
            logits = outputs.logits
            predictions = torch.argmax(logits, dim=-1)

            all_preds.extend(predictions.cpu().numpy())
            all_labels.extend(batch["labels"].cpu().numpy())

    macro_f1 = f1_score(all_labels, all_preds, average='macro')
    model.train() # Set model back to training mode
    return macro_f1

# Example usage:
macro_f1_score = evaluate_model(model, tokenizer, test_data, device)
print(f"Macro F1 Score on test data: {macro_f1_score}")

Macro F1 Score on test data: 0.9566723259762309
